In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
from celavi.data_manager import PVTechUnitLocations, PVTechUnitChars, StandardScenarios
from celavi.compute_locations import ComputeLocations

In [3]:
start_year = 2010

In [4]:
chars = Path('C:/Users/rhanes/GitHub/celavi-data/inputs_to_preprocessing/glasspermodule_pvice.csv')
locs = Path('C:/Users/rhanes/GitHub/celavi-data/inputs_to_preprocessing/uspvdb_v1_0_20231108.csv')
fac_type = Path('C:/Users/rhanes/GitHub/celavi-data/inputs/facility_type.csv')
stscen_file = Path('C:/Users/rhanes/GitHub/celavi-data/inputs_to_preprocessing/StScen20A_MidCase_annual_state.csv')

facility_type_lookup = pd.read_csv(fac_type, header=None)

In [5]:
# Process data for solar power plants - from USPVDB
pv_locs_raw = PVTechUnitLocations(fpath = locs, backfill = True)

validating PVTechUnitLocations
validated PVTechUnitLocations
no missing data values in celavi.data_manager.PVTechUnitLocations.eia_id
no missing data values in celavi.data_manager.PVTechUnitLocations.p_year


In [32]:
# select only those plants with eia_ids 
pv_locs = pv_locs_raw[(pv_locs_raw['eia_id'] != '-1') & (pv_locs_raw['p_year'] != '-1')]

# reformat data for later use
pv_locs = pv_locs.rename(
    columns={
        "p_state": "region_id_2",
        'p_county': 'region_id_3',
        "xlong": "long",
        "ylat": "lat",
        "eia_id": "facility_id",
        "p_year": "year"
        },
        )

# exclude Hawaii, Guam, Puerto Rico, and Alaska (only have road network data for the contiguous United States)
pv_locs.drop(
    index = pv_locs[pv_locs.region_id_2.isin(['HI','GU','PR','AK'])].index,
    inplace = True
)

# exclude Nantucket since transport routing doesn't currently include ferries
pv_locs.drop(
    index = pv_locs[pv_locs.region_id_3 == 'Nantucket'].index,
    inplace = True
)

# exclude all PV subtypes other than c-Si
# that's the only one we have glass information on
pv_locs.drop(
    index = pv_locs[pv_locs.p_tech_sec != 'c-si'].index,
    inplace = True
)

# then the tech_sec column is no longer needed
pv_locs.drop(
    columns = 'p_tech_sec',
    inplace = True
)

# also drop data from before the simulation start year
pv_locs.drop(
    index = pv_locs[pv_locs.year < start_year].index,
    inplace = True
)

# Aggregate to STATE level by SUMMING capacity and AVERAGING
# location. State = region_id_2
# (this is the plant location for each facility_id)
fac_ids_state = pv_locs[['facility_id','region_id_2']].drop_duplicates(
    subset='region_id_2', keep='last'
    )
pv_locs_state = pv_locs.groupby(
    ['region_id_2','year']
    ).agg(
        {'lat': np.mean, 'long': np.mean, 'p_cap_dc': 'sum'}
        ).reset_index(
        ).merge(
            fac_ids_state, on='region_id_2', how='outer'
            )

# Filter down the dataset to generate the number_of_technology_units
# file
# Store this dataframe into self for use in capacity projection
# calculations and creation of the number_of_technology_units file
capacity_data = pv_locs_state[
    ['facility_id', 'region_id_2', 'year', 'p_cap_dc']
    ].drop_duplicates().dropna()

In [80]:
stscen = StandardScenarios(fpath = stscen_file, backfill = True).rename(
            columns={'t': 'year'}
        )
modules = PVTechUnitChars(fpath=chars, backfill=True)

# group stscen by state and take the consecutive difference of the
# capacity column to get new MW-dc installations by year
stscen['cap_new'] = stscen.groupby('state')['upv_MW'].diff()

# where total capacity decreases in a year, set the new capacity value
# to 0
stscen.loc[stscen['cap_new'] < 0,'cap_new'] = 0

# .diff() leaves empty values where there is no previous row.
# replace these NAs with 0
stscen.fillna(value=0, inplace=True)

# merge the average capacity extrapolation with the standard scenario
# data by year
# keep only the columns required to calculate the number of new
# turbines
joined = modules[modules.year > 2020].merge(
    stscen[stscen.year > 2020],
    on='year',
    how='outer',
    suffixes=('avgcap', 'stdscen'),
    sort=True
)[['year', 'state', 'MWdc_per_module', 'cap_new']]

# calculate the number of new modules by dividing the new capacity
# addition with the average module capacity
# round up to the nearest integer to deal in whole numbers of modules
joined['n_module'] = np.ceil(joined.cap_new / joined.MWdc_per_module)

# remove any entries where no new turbines are installed
joined = joined[joined.n_module > 0]

# generate project names for the future capacity
joined['p_name'] = joined.state + '_future_cap'

# remove columns no longer needed
capacity_future = joined.copy()[['year', 'state', 'p_name', 'n_module']]

capacity_future.rename(
    columns = {'state':'region_id_2'},
    inplace = True
)

capacity_future

validating StandardScenarios
validated StandardScenarios
no missing data values in celavi.data_manager.StandardScenarios.wind-ons_MW
no missing data values in celavi.data_manager.StandardScenarios.upv_MW
validating PVTechUnitChars
validated PVTechUnitChars
no missing data values in celavi.data_manager.PVTechUnitChars.year
no missing data values in celavi.data_manager.PVTechUnitChars.MWdc_per_m2
no missing data values in celavi.data_manager.PVTechUnitChars.MWdc_per_module
no missing data values in celavi.data_manager.PVTechUnitChars.glass_metrictonne_per_module


,year,region_id_2,p_name,n_module
1,2022.0,TX,TX_future_cap,1789100.0
3,2024.0,TX,TX_future_cap,17814317.0
5,2026.0,TX,TX_future_cap,13871546.0
7,2028.0,TX,TX_future_cap,7781989.0
9,2030.0,TX,TX_future_cap,17285103.0
11,2032.0,TX,TX_future_cap,2670944.0
13,2034.0,TX,TX_future_cap,3264000.0
15,2036.0,TX,TX_future_cap,3802946.0
17,2038.0,TX,TX_future_cap,13180251.0
21,2042.0,TX,TX_future_cap,37704419.0


In [71]:
capacity_unit_counts = capacity_data.merge(
    modules[['year','MWdc_per_module']], on='year', how='outer'
).dropna(
).sort_values(
    by=['region_id_2', 'year']
)

# Take sequential differences in installed capacity within each state to
# calcualte new yearly installations
# (the sort above is necesasry for this to work properly)
capacity_unit_counts['cap_new'] = capacity_unit_counts.groupby('region_id_2')['p_cap_dc'].diff().replace({np.nan: None})

# New installations for the first year a state appears in the data is set as the observed
# installed capacity for that year (a simplification, but this lets us capture that initial
# capacity so we don't under-count)
_replace_index = capacity_unit_counts[capacity_unit_counts.cap_new.values == None]['cap_new'].index
capacity_unit_counts.loc[_replace_index, 'cap_new'] = capacity_unit_counts.p_cap_dc[_replace_index]

capacity_unit_counts['n_module'] = np.ceil(capacity_unit_counts.cap_new / capacity_unit_counts.MWdc_per_module)

capacity_unit_counts.loc[capacity_unit_counts.n_module < 0, 'n_module'] = 0.0

capacity_unit_counts['p_name'] = capacity_unit_counts.region_id_2 + '_hist'

capacity_unit_counts.drop(
    columns = ['p_cap_dc','MWdc_per_module','cap_new'],
    inplace = True
)

capacity_unit_counts

# Why don't we need region_id in capacity data?
# because it's joined elsewhere

,facility_id,region_id_2,year,n_module,p_name
0,59862.0,AL,2015.0,122059,AL_hist
24,59862.0,AL,2016.0,167715,AL_hist
57,59862.0,AL,2017.0,82486,AL_hist
25,62683.0,AR,2016.0,4858,AR_hist
58,62683.0,AR,2017.0,17797,AR_hist
...,...,...,...,...,...
94,60893.0,WI,2017.0,51695,WI_hist
127,60893.0,WI,2018.0,0.0,WI_hist
199,60893.0,WI,2020.0,567500,WI_hist
305,60893.0,WI,2021.0,0.0,WI_hist


In [ ]:

"""
data = {'feature1_in_use?': [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1],
     'feature1_available?': [0, 1, 1, np.nan, 1, np.nan, 1, 1, np.nan, 1, 1]}

df = pd.DataFrame(data)
df['feature1_available?'] = df['feature1_available?'].replace({np.nan : None})
condition_list = [
    ((df['feature1_available?'].values == None) & (df['feature1_in_use?'] == 1)),
    ((df['feature1_available?'].values == None) & (df['feature1_in_use?'] == 0))
]
choice_list = [df['feature1_in_use?'], 'X']
df['feature1_available?'] = np.select(condition_list, choice_list, df['feature1_available?'])
df
"""

In [ ]:


# Use the computed locations dataset to generate unique facility_id
# values for these future "power plants"
_facility_id_start = int(self.locs.facility_id.max() + 1)

# generate a list of new facility IDs
_new_facility_id = list(
    _facility_id_start +
    np.arange(len(capacity_future.p_name.unique()))
)

# create a data frame of new facility IDs and project names,
# for merging
_new_facility_id = pd.DataFrame(
    data={
        'p_name': list(capacity_future.p_name.unique()),
        'facility_id': _new_facility_id
    }
)

# merge to add a facility_id column to the capacity projection data
capacity_future = capacity_future.merge(
    _new_facility_id[['p_name', 'facility_id']],
    on='p_name',
    how='outer'
)
self.capacity_data = pd.concat([self.capacity_data,capacity_future])
self.capacity_data = self.capacity_data.sort_values(by = list(self.capacity_data.columns)).rename(
    columns={'n_turbine': 'n_technology'}
).to_csv(
    self.technology_data_filename,
    index=False
)


# get state column back by splitting p_name
# ig1 and ig2 are dummy columns not used further
_new_facility_id[
    ['region_id_2', 'ig1', 'ig2']
] = _new_facility_id.p_name.str.split('_',
                                        expand=True)

# Calculate lat/long pairs for the future power plants by taking the
# average lat/long of existing power plants by state
_new_facility_locs = _new_facility_id[
    ['facility_id', 'region_id_2']
].merge(
    self.locs.groupby(
        by='region_id_2'
    ).mean(
        ['lat','long']
    ).reset_index()[['region_id_2', 'lat', 'long']],
    on='region_id_2',
    how='left'
)

_new_facility_locs['facility_type'] = 'power plant'
_new_facility_locs['region_id_1'] = 'USA'
_new_facility_locs['region_id_3'] = ''
_new_facility_locs['region_id_4'] = ''

# Add the future power plants to the locations dataset stored in self
# It has to go back into self to get saved at the end of the
# join_facilities method
self.locs = pd.concat([self.locs,_new_facility_locs])
self.locs = self.locs.sort_values(by = list(self.locs.columns))